In [ ]:
from pathlib import Path
import os
import sys

import mne
import numpy as np

project_root = next(
    path for path in [Path.cwd(), *Path.cwd().parents]
    if (path / "pyproject.toml").exists()
)

os.chdir(project_root)

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

%matplotlib qt

In [ ]:
from modules.decode_trigger import decode_8bit_trigger, convert_dict_trigger
from modules.events import get_events_tms_per_task
from modules.ica import ICAProcessor

In [ ]:
raw_data = mne.io.read_raw_bdf(r"data/raw/V1.bdf", preload=True)

In [ ]:
emg_ch_names = ["EMG L", "EMG R"]
eog_ch_names = ["EOG"]
raw_data.set_channel_types({ch: "emg" for ch in emg_ch_names})
raw_data.set_channel_types({'EOG':'eog'})

montage = mne.channels.make_standard_montage("standard_1020",head_size='auto')
raw_data.set_montage(montage)

In [ ]:
raw_data.drop_channels(emg_ch_names)

In [ ]:
bad_ch =['F7', 'F5', 'P7', 'CPz', 'Oz', 'Iz', 'PO4', 'PO8', 'O2', "P6", "P8", "TP8", "C6", "TP10", "TP9", "F6", "FT8", "T8", "T7", "F3", "FC5", "FC3", "C3", "C1"]
raw_data.drop_channels(bad_ch)

In [26]:
raw_data = mne.preprocessing.fix_stim_artifact(
    raw_data, 
    events=events,
    event_id=7,
    tmin=-0.004,  # -2 ms
    tmax=0.018, # 10 ms
    mode='linear'
)

In [ ]:
filtered_data = raw_data.copy().filter(l_freq=1, h_freq=55, method='fir')

In [ ]:
events, event_id = mne.events_from_annotations(raw_data)
event_id = raw_data.event_id = convert_dict_trigger(event_id, decode_8bit_trigger)

In [35]:
events_tms, events_id_tms = get_events_tms_per_task(events, event_id)

In [ ]:
epochs = mne.Epochs(
    filtered_data,
    events=events_tms,
    event_id=events_id_tms,
    tmin=-2,
    tmax=0.05,
    preload=True,
    baseline=None
)

In [ ]:
epochs.resample(500)

In [ ]:
epochs.plot()

In [ ]:
from autoreject import get_rejection_threshold
reject_thresholds = get_rejection_threshold(
    epochs, 
    ch_types='eeg', 
    decim=2, 
    random_state=42 # Para reprodutibilidade
)

print(f"Limiares ideais calculados: {reject_thresholds}")

In [ ]:
epochs.drop_bad(reject=reject_thresholds)

In [46]:
ica = ICAProcessor()

In [ ]:
ica.fit(epochs)

In [17]:
ica.plot_components()

Not setting metadata
100 matching events found
No baseline correction applied
0 projection items activated


In [ ]:
exclide_comp = ica.get_exclude_components()

In [ ]:
epochs_clean = ica.apply_ica(exclude=exclide_comp)

Applying ICA to Epochs instance
    Transforming to ICA space (40 components)
    Zeroing out 35 ICA components
    Projecting back using 40 PCA components


In [ ]:
epochs_clean.plot()

In [ ]:
epochs_clean.set_eeg_reference('average', projection=True)

In [ ]:
epochs_clean.save("data/processed/V1/epochs-mvar.fif", overwrite=True)